# 04 — Parallel Execution

Fan-out starts independent work from common state. Fan-in synchronizes the required branches and merges their updates before downstream execution continues.

## Topology

```mermaid
flowchart LR
    accTitle: Parallel Source Collection
    accDescr: Dispatch fans out to three independent sources whose results synchronize at aggregation before the graph ends.

    start([START]) --> dispatch[Dispatch]
    dispatch --> source_a[Source A]
    dispatch --> source_b[Source B]
    dispatch --> source_c[Source C]
    source_a --> aggregate[Aggregate]
    source_b --> aggregate
    source_c --> aggregate
    aggregate --> finish([END])
```

Parallelism reduces wall-clock latency only when branches are independent and can actually run concurrently. It does not reduce total work.

## Parallel state needs reducer semantics

All three source nodes update `results`. Without a reducer, those concurrent writes conflict. List addition preserves every contribution, while stable source identifiers let fan-in sort without depending on completion order.

In [6]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class ParallelState(TypedDict, total=False):
    question: str
    results: Annotated[list[tuple[str, str]], operator.add]
    errors: Annotated[list[tuple[str, str]], operator.add]
    evidence: list[str]
    trace: Annotated[list[str], operator.add]

## Independent source nodes

Each source reads only the question and returns its own labeled result. In a real graph these could be independent APIs or retrievers with per-branch timeout and retry policy.

In [7]:
def dispatch(_: ParallelState) -> dict:
    return {"trace": ["dispatch"]}


def source_a(state: ParallelState) -> dict:
    return {"results": [("a", f"Primary evidence for: {state['question']}")], "trace": ["source:a"]}


def source_b(state: ParallelState) -> dict:
    return {"results": [("b", f"Market context for: {state['question']}")], "trace": ["source:b"]}


def source_c(state: ParallelState) -> dict:
    return {"results": [("c", f"Policy limitation for: {state['question']}")], "trace": ["source:c"]}

## Fan-in is a synchronization boundary

The aggregate node runs after all listed branches complete. It converts reducer output into deterministic downstream state.

In [8]:
def aggregate(state: ParallelState) -> dict:
    ordered = sorted(state["results"], key=lambda item: item[0])
    return {
        "evidence": [text for _, text in ordered],
        "trace": [f"aggregate:{len(ordered)}"],
    }

## Build fan-out and fan-in

Three outgoing edges create fan-out. Passing the three source names together as the start of one edge creates a fan-in barrier before aggregation.

In [9]:
builder = StateGraph(ParallelState)
builder.add_node("dispatch", dispatch)
builder.add_node("source_a", source_a)
builder.add_node("source_b", source_b)
builder.add_node("source_c", source_c)
builder.add_node("aggregate", aggregate)

builder.add_edge(START, "dispatch")
builder.add_edge("dispatch", "source_a")
builder.add_edge("dispatch", "source_b")
builder.add_edge("dispatch", "source_c")
builder.add_edge(["source_a", "source_b", "source_c"], "aggregate")
builder.add_edge("aggregate", END)

graph = builder.compile()

In [10]:
result = graph.invoke(
    {"question": "How can heatwaves affect electricity demand?", "results": [], "errors": [], "trace": []}
)

assert sorted(source for source, _ in result["results"]) == ["a", "b", "c"]
assert len(result["evidence"]) == 3
assert result["trace"][-1] == "aggregate:3"
print(*result["evidence"], sep="\n- ")

Primary evidence for: How can heatwaves affect electricity demand?
- Market context for: How can heatwaves affect electricity demand?
- Policy limitation for: How can heatwaves affect electricity demand?


## Partial failure policy

A production fan-in must decide whether it requires all branches, a quorum, or any success. One safe design lets source nodes return labeled `errors` rather than silently dropping work; the aggregate node then applies an explicit coverage policy. Other systems may retry only the failed branch or abort the whole run.

State conflicts, cancellation, concurrency limits, and per-branch timeouts are part of parallel design—not incidental syntax.

Next: Notebook 05 separates semantic and runtime failures and applies bounded recovery.